# RoadGuard Qro — 01: Data Preparation & Stratified YOLO Conversion
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AlarconCastilloDiegoAndre/RoadGuard-Qro/blob/main/ml/01_prepare_data.ipynb)

This notebook downloads the **Road Damage Dataset (RDD2022)**, filters target classes (`D00`, `D10`, `D20`, `D40`), and creates a stratified 70/20/10 train/val/test split in standard Ultralytics YOLO format.


In [ ]:
# Step 1: Install dependencies and clone repository if running in Colab
import os
import sys

if 'google.colab' in str(get_ipython()):
    print("Running in Google Colab. Setting up workspace...")
    !git clone https://github.com/AlarconCastilloDiegoAndre/RoadGuard-Qro.git
    %cd RoadGuard-Qro
    !pip install -q -r ml/requirements.txt
else:
    print("Running in local environment.")


In [ ]:
# Step 2: Download RDD2022 from official Figshare repository (CC BY 4.0)
import requests
from pathlib import Path

DATA_DIR = Path("ml/data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)
zip_path = DATA_DIR / "RDD2022.zip"

FIGSHARE_URL = "https://ndownloader.figshare.com/files/38030910"

if not zip_path.exists():
    print(f"Downloading RDD2022 dataset (~13.2 GB)...")
    !curl -L -o {str(zip_path)} {FIGSHARE_URL}
else:
    print(f"Dataset archive already exists at {zip_path}")


In [ ]:
# Step 3: Extract archive
EXTRACT_DIR = Path("ml/data/raw/RDD2022_extracted")
if not EXTRACT_DIR.exists():
    print("Extracting dataset...")
    !unzip -q {str(zip_path)} -d {str(EXTRACT_DIR)}
    print("Extraction completed.")
else:
    print("Extracted files already exist.")


In [ ]:
# Step 4: Run stratified conversion to YOLO format
from ml.scripts.convert_rdd_to_yolo import convert_dataset
from pathlib import Path

source_path = EXTRACT_DIR / "RDD2022" if (EXTRACT_DIR / "RDD2022").exists() else EXTRACT_DIR
dest_path = Path("ml/data/yolo_rdd")

print(f"Converting annotations from {source_path} to {dest_path}...")
counts = convert_dataset(
    source_root=source_path,
    output_root=dest_path,
    train_ratio=0.70,
    val_ratio=0.20,
    test_ratio=0.10,
    seed=42
)


In [ ]:
# Step 5: Visualize sample converted bounding boxes
import cv2
import matplotlib.pyplot as plt
import yaml

with open(dest_path / "rdd2022.yaml", "r") as f:
    config = yaml.safe_load(f)

print("YOLO Configuration:")
print(yaml.dump(config))

# Plot 4 sample images with ground truth boxes
sample_labels = list((dest_path / "labels" / "train").glob("*.txt"))[:4]
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, lf in zip(axes.flatten(), sample_labels):
    img_file = dest_path / "images" / "train" / (lf.stem + ".jpg")
    if not img_file.exists():
        continue
    img = cv2.imread(str(img_file))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w, _ = img.shape

    with open(lf) as f:
        for line in f:
            cid, cx, cy, bw, bh = map(float, line.strip().split())
            xmin = int((cx - bw / 2) * w)
            ymin = int((cy - bh / 2) * h)
            xmax = int((cx + bw / 2) * w)
            ymax = int((cy + bh / 2) * h)
            cv2.rectangle(img, (xmin, ymin), (xmax, ymax), (255, 0, 0), 3)
            cname = config['names'].get(int(cid), str(cid))
            cv2.putText(img, cname, (xmin, max(20, ymin - 10)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)

    ax.imshow(img)
    ax.set_title(lf.stem)
    ax.axis("off")

plt.tight_layout()
plt.show()
